## 1) Imports and folders

In [11]:
import os
import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pdfplumber
import pytesseract
import cv2
from PIL import Image
import joblib

from IPython.display import display

PROJECT_DIR = Path.cwd()
SAVED_MODELS_DIR = PROJECT_DIR / "saved_models"
UPLOADS_DIR = PROJECT_DIR / "uploads"
OUTPUTS_DIR = PROJECT_DIR / "report_outputs"

for d in [SAVED_MODELS_DIR, UPLOADS_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Saved models:", SAVED_MODELS_DIR)
print("Uploads:", UPLOADS_DIR)
print("Outputs:", OUTPUTS_DIR)

Project: C:\Users\Rutik Bhendarkar\Chat Bot 2
Saved models: C:\Users\Rutik Bhendarkar\Chat Bot 2\saved_models
Uploads: C:\Users\Rutik Bhendarkar\Chat Bot 2\uploads
Outputs: C:\Users\Rutik Bhendarkar\Chat Bot 2\report_outputs


## 2) OCR helpers

In [12]:
def preprocess_image_for_ocr(image):
    arr = np.array(image)
    if arr.ndim == 3:
        gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    else:
        gray = arr
    gray = cv2.fastNlMeansDenoising(gray, None, 30, 7, 21)
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    return cv2.filter2D(thresh, -1, kernel)

def clean_text(text):
    text = text.replace("\x0c", "\n")
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" +\n", "\n", text)
    return text.strip()

def extract_text_from_pdf(pdf_path):
    parts = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            txt = page.extract_text()
            if txt and txt.strip():
                parts.append(f"--- Page {i} ---\n{txt}")
            else:
                img = page.to_image(resolution=350).original
                processed = preprocess_image_for_ocr(img)
                ocr_txt = pytesseract.image_to_string(processed, config="--psm 6")
                parts.append(f"--- Page {i} (OCR) ---\n{ocr_txt}")
    return clean_text("\n\n".join(parts))

def extract_text_from_image(image_path):
    img = Image.open(image_path).convert("RGB")
    processed = preprocess_image_for_ocr(img)
    txt = pytesseract.image_to_string(processed, config="--psm 6")
    return clean_text(txt)

## 3) Report type detection

In [13]:
def detect_report_type(text):
    t = text.lower()
    scores = {
        "diabetes": sum(k in t for k in ["glucose", "blood sugar", "hba1c", "insulin", "fasting glucose"]),
        "heart": sum(k in t for k in ["cholesterol", "ecg", "troponin", "angina", "heart rate", "chest pain"]),
        "liver": sum(k in t for k in ["bilirubin", "alt", "ast", "albumin", "alkaline phosphatase", "sgpt", "sgot"]),
        "cbc": sum(k in t for k in ["hemoglobin", "wbc", "rbc", "platelet", "hematocrit", "mcv", "mch", "mchc", "rdw"]),
    }
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "unknown"

def report_type_reason(report_type):
    return {
        "diabetes": "Glucose / insulin terms detected.",
        "heart": "Heart / cholesterol / ECG terms detected.",
        "liver": "Bilirubin / ALT / AST / albumin terms detected.",
        "cbc": "Blood count / CBC terms detected.",
        "unknown": "Could not confidently identify the report."
    }.get(report_type, "Unknown report type.")

## 4) Flexible regex extraction

In [14]:
def first_match(patterns, text, cast=float):
    for pattern in patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            try:
                return cast(m.group(1))
            except Exception:
                pass
    return None

def extract_report_fields(text, report_type):
    num = r"([0-9]+(?:\.[0-9]+)?)"
    integer = r"([0-9]+)"
    report = {}

    if report_type == "diabetes":
        report["Glucose"] = first_match([rf"glucose\s*[:=\-]?\s*{num}",
                                         rf"blood\s*sugar\s*[:=\-]?\s*{num}",
                                         rf"fasting\s*glucose\s*[:=\-]?\s*{num}"], text)
        report["BMI"] = first_match([rf"bmi\s*[:=\-]?\s*{num}"], text)
        report["BloodPressure"] = first_match([rf"blood\s*pressure\s*[:=\-]?\s*{num}",
                                               rf"bp\s*[:=\-]?\s*{num}"], text)
        report["Insulin"] = first_match([rf"insulin\s*[:=\-]?\s*{num}"], text)
        report["Age"] = first_match([rf"age\s*[:=\-]?\s*{integer}"], text, cast=int)

    elif report_type == "liver":
        report["Age"] = first_match([rf"age\s*[:=\-]?\s*{integer}"], text, cast=int)
        report["Gender"] = re.search(r"gender\s*[:=\-]?\s*(male|female)", text, re.I)
        report["Gender"] = report["Gender"].group(1).lower() if report["Gender"] else None
        report["Total_Bilirubin"] = first_match([rf"total\s*bilirubin\s*[:=\-]?\s*{num}",
                                                 rf"bilirubin\s*[:=\-]?\s*{num}"], text)
        report["Direct_Bilirubin"] = first_match([rf"direct\s*bilirubin\s*[:=\-]?\s*{num}"], text)
        report["Alkaline_Phosphotase"] = first_match([rf"alkaline\s*phosphotase\s*[:=\-]?\s*{num}",
                                                      rf"alkaline\s*phosphatase\s*[:=\-]?\s*{num}"], text)
        report["Alamine_Aminotransferase"] = first_match([rf"alamine\s*aminotransferase\s*[:=\-]?\s*{num}",
                                                          rf"alt\s*[:=\-]?\s*{num}",
                                                          rf"sgpt\s*[:=\-]?\s*{num}"], text)
        report["Aspartate_Aminotransferase"] = first_match([rf"aspartate\s*aminotransferase\s*[:=\-]?\s*{num}",
                                                            rf"ast\s*[:=\-]?\s*{num}",
                                                            rf"sgot\s*[:=\-]?\s*{num}"], text)
        report["Total_Protiens"] = first_match([rf"total\s*protiens\s*[:=\-]?\s*{num}",
                                                rf"total\s*proteins\s*[:=\-]?\s*{num}"], text)
        report["Albumin"] = first_match([rf"albumin\s*[:=\-]?\s*{num}"], text)
        report["Albumin_and_Globulin_Ratio"] = first_match([rf"albumin\s*and\s*globulin\s*ratio\s*[:=\-]?\s*{num}",
                                                            rf"a\/g\s*ratio\s*[:=\-]?\s*{num}"], text)

    elif report_type == "heart":
        report["age"] = first_match([rf"age\s*[:=\-]?\s*{integer}"], text, cast=int)
        m = re.search(r"cp\s*[:=\-]?\s*([a-zA-Z\s\-]+)", text, re.I)
        report["cp"] = m.group(1).strip() if m else None
        report["trestbps"] = first_match([rf"trestbps\s*[:=\-]?\s*{num}",
                                          rf"resting\s*blood\s*pressure\s*[:=\-]?\s*{num}"], text)
        report["chol"] = first_match([rf"chol\s*[:=\-]?\s*{num}",
                                      rf"cholesterol\s*[:=\-]?\s*{num}"], text)
        report["thalch"] = first_match([rf"thalch\s*[:=\-]?\s*{num}"], text)
        report["oldpeak"] = first_match([rf"oldpeak\s*[:=\-]?\s*{num}"], text)
        report["ca"] = first_match([rf"ca\s*[:=\-]?\s*{integer}"], text, cast=int)
        m = re.search(r"thal\s*[:=\-]?\s*([a-zA-Z\s\-]+)", text, re.I)
        report["thal"] = m.group(1).strip() if m else None
        m = re.search(r"exang\s*[:=\-]?\s*(true|false|0|1)", text, re.I)
        report["exang"] = m.group(1).lower() if m else None

    elif report_type == "cbc":
        report["Age"] = first_match([rf"age\s*[:=\-]?\s*{integer}"], text, cast=int)
        report["Hemoglobin"] = first_match([rf"hemoglobin\s*[:=\-]?\s*{num}",
                                            rf"hb\s*[:=\-]?\s*{num}"], text)
        report["WBC"] = first_match([rf"wbc\s*[:=\-]?\s*{num}",
                                     rf"white\s*blood\s*cell\s*count\s*[:=\-]?\s*{num}"], text)
        report["RBC"] = first_match([rf"rbc\s*[:=\-]?\s*{num}"], text)
        report["Platelets"] = first_match([rf"platelets?\s*[:=\-]?\s*{num}",
                                           rf"plt\s*[:=\-]?\s*{num}"], text)
        report["Hematocrit"] = first_match([rf"hematocrit\s*[:=\-]?\s*{num}",
                                            rf"hct\s*[:=\-]?\s*{num}"], text)
        report["MCV"] = first_match([rf"mcv\s*[:=\-]?\s*{num}"], text)
        report["MCH"] = first_match([rf"mch\s*[:=\-]?\s*{num}"], text)
        report["MCHC"] = first_match([rf"mchc\s*[:=\-]?\s*{num}"], text)
        report["RDW"] = first_match([rf"rdw\s*[:=\-]?\s*{num}"], text)

    return {k: v for k, v in report.items() if v is not None}

## 5) Load saved models

In [15]:
def load_first_existing(candidates):
    for name in candidates:
        path = SAVED_MODELS_DIR / name
        if path.exists():
            try:
                obj = joblib.load(path)
                print("Loaded:", name)
                return obj, str(path)
            except Exception as e:
                print("Failed:", name, e)
    return None, None

MODELS = {
    "diabetes_model": load_first_existing(["diabetes_model.pkl", "diabetes_model_pipeline.pkl", "diabetes_xgb_model.pkl", "xgb_model.pkl"])[0],
    "heart_model": load_first_existing(["heart_model_pipeline.pkl", "heart_model.pkl"])[0],
    "liver_model": load_first_existing(["liver_model_pipeline.pkl", "liver_model.pkl"])[0],
    "cbc_model": load_first_existing(["cbc_xgb_model.pkl"])[0],
    "cbc_scaler": load_first_existing(["cbc_scaler.pkl"])[0],
    "cbc_feature_columns": load_first_existing(["cbc_feature_columns.pkl"])[0],
    "cbc_feature_medians": load_first_existing(["cbc_feature_medians.pkl"])[0],
    "cbc_label_encoder": load_first_existing(["cbc_label_encoder.pkl"])[0],
}
print("Models ready.")

Loaded: xgb_model.pkl
Loaded: heart_model_pipeline.pkl
Loaded: liver_model_pipeline.pkl
Loaded: cbc_xgb_model.pkl
Loaded: cbc_scaler.pkl
Loaded: cbc_feature_columns.pkl
Loaded: cbc_feature_medians.pkl
Loaded: cbc_label_encoder.pkl
Models ready.


## 6) Rule engines and prediction helpers

In [16]:
def safe_prob(p):
    try:
        return float(np.round(p * 100, 2))
    except Exception:
        return None

def predict_pipeline_model(model, row_dict):
    if model is None:
        return None
    try:
        X = pd.DataFrame([row_dict])
        pred = int(model.predict(X)[0])
        prob = None
        if hasattr(model, "predict_proba"):
            prob = safe_prob(model.predict_proba(X)[0][1])
        return {"prediction": pred, "probability": prob}
    except Exception as e:
        print("Prediction error:", e)
        return None

def predict_cbc_model(models, row_dict):
    model = models["cbc_model"]
    cols = models["cbc_feature_columns"]
    meds = models["cbc_feature_medians"]
    scaler = models["cbc_scaler"]
    if model is None or cols is None or meds is None:
        return None
    try:
        input_row = {c: meds.get(c, 0) for c in cols}
        for k, v in row_dict.items():
            if k in input_row:
                input_row[k] = v
        X = pd.DataFrame([input_row])[cols]
        X = scaler.transform(X) if scaler is not None else X
        pred = int(model.predict(X)[0])
        prob = None
        if hasattr(model, "predict_proba"):
            prob = safe_prob(model.predict_proba(X)[0][1])
        return {"prediction": pred, "probability": prob}
    except Exception as e:
        print("CBC prediction error:", e)
        return None

def diabetes_rule_engine(r):
    score = 0
    findings = []
    def add(p, s, m, pts):
        nonlocal score
        findings.append({"parameter": p, "severity": s, "message": m})
        score += pts
    g = r.get("Glucose")
    if g is not None:
        if g >= 200: add("Glucose", "CRITICAL", "Very high glucose detected.", 30)
        elif g >= 140: add("Glucose", "HIGH", "High glucose detected.", 20)
    bmi = r.get("BMI")
    if bmi is not None and bmi >= 30: add("BMI", "MODERATE", "High BMI detected.", 10)
    bp = r.get("BloodPressure")
    if bp is not None and bp >= 140: add("Blood Pressure", "HIGH", "High blood pressure detected.", 15)
    ins = r.get("Insulin")
    if ins is not None and ins >= 100: add("Insulin", "MODERATE", "Elevated insulin detected.", 8)
    overall = "CRITICAL" if score >= 60 else "HIGH" if score >= 40 else "MODERATE" if score >= 20 else "LOW"
    rec = []
    if g is not None and g >= 140: rec.append("Monitor blood sugar and reduce sugar intake.")
    if bmi is not None and bmi >= 30: rec.append("Exercise and maintain a healthy diet.")
    if bp is not None and bp >= 140: rec.append("Monitor blood pressure and reduce salt intake.")
    if not rec: rec.append("Continue healthy habits and regular checkups.")
    summary = {
        "CRITICAL": "Critical diabetes risk pattern detected.",
        "HIGH": "High diabetes risk pattern detected.",
        "MODERATE": "Some diabetes risk markers are present.",
        "LOW": "No major diabetes-risk pattern detected.",
    }[overall]
    return {"findings": findings, "risk_score": score, "overall_risk": overall, "summary": summary, "recommendations": rec}

def liver_rule_engine(r):
    score = 0
    findings = []
    def add(p, s, m, pts):
        nonlocal score
        findings.append({"parameter": p, "severity": s, "message": m})
        score += pts
    tb = r.get("Total_Bilirubin")
    if tb is not None and tb >= 1.5: add("Total Bilirubin", "HIGH", "Raised bilirubin detected.", 16)
    alt = r.get("Alamine_Aminotransferase")
    if alt is not None and alt >= 40: add("ALT", "MODERATE", "ALT is elevated.", 8)
    ast = r.get("Aspartate_Aminotransferase")
    if ast is not None and ast >= 40: add("AST", "MODERATE", "AST is elevated.", 8)
    alb = r.get("Albumin")
    if alb is not None and alb < 3.0: add("Albumin", "HIGH", "Albumin is low.", 14)
    ag = r.get("Albumin_and_Globulin_Ratio")
    if ag is not None and ag < 1.0: add("A/G Ratio", "MODERATE", "Low albumin/globulin ratio.", 8)
    overall = "CRITICAL" if score >= 60 else "HIGH" if score >= 40 else "MODERATE" if score >= 20 else "LOW"
    rec = []
    if tb is not None and tb >= 1.5: rec.append("Avoid alcohol and follow liver-friendly habits.")
    if alt is not None and alt >= 40 or ast is not None and ast >= 40: rec.append("Repeat liver tests and consult a doctor.")
    if alb is not None and alb < 3.0: rec.append("Check nutrition and liver function regularly.")
    if not rec: rec.append("Maintain a healthy lifestyle and routine checkups.")
    summary = {
        "CRITICAL": "Critical liver risk pattern detected.",
        "HIGH": "High liver risk pattern detected.",
        "MODERATE": "Some liver-related changes are present.",
        "LOW": "No major liver-risk pattern detected.",
    }[overall]
    return {"findings": findings, "risk_score": score, "overall_risk": overall, "summary": summary, "recommendations": rec}

def heart_rule_engine(r):
    score = 0
    findings = []
    def add(p, s, m, pts):
        nonlocal score
        findings.append({"parameter": p, "severity": s, "message": m})
        score += pts
    age = r.get("age")
    if age is not None and age >= 60: add("Age", "MODERATE", "Older age increases heart risk.", 8)
    cp = str(r.get("cp", "")).lower()
    if "typical angina" in cp: add("Chest Pain Type", "CRITICAL", "Typical angina is a serious warning sign.", 25)
    elif "atypical angina" in cp: add("Chest Pain Type", "HIGH", "Atypical angina may indicate heart disease.", 18)
    trestbps = r.get("trestbps")
    if trestbps is not None and trestbps >= 140: add("Resting BP", "HIGH", "High resting blood pressure detected.", 15)
    chol = r.get("chol")
    if chol is not None and chol >= 240: add("Cholesterol", "HIGH", "High cholesterol detected.", 12)
    exang = str(r.get("exang", "")).lower()
    if exang in ["true", "1"]: add("Exercise Angina", "HIGH", "Exercise-induced angina detected.", 15)
    oldpeak = r.get("oldpeak")
    if oldpeak is not None and oldpeak >= 1.0: add("Oldpeak", "MODERATE", "ST depression detected.", 8)
    overall = "CRITICAL" if score >= 55 else "HIGH" if score >= 35 else "MODERATE" if score >= 18 else "LOW"
    rec = []
    if trestbps is not None and trestbps >= 140: rec.append("Monitor blood pressure and reduce salt intake.")
    if chol is not None and chol >= 240: rec.append("Maintain a heart-friendly diet and regular exercise.")
    if exang in ["true", "1"]: rec.append("Consult a doctor if chest pain continues.")
    if not rec: rec.append("Continue healthy habits and regular checkups.")
    summary = {
        "CRITICAL": "Critical heart-risk pattern detected.",
        "HIGH": "High heart-risk pattern detected.",
        "MODERATE": "Some heart-risk markers are present.",
        "LOW": "No major heart-risk pattern detected.",
    }[overall]
    return {"findings": findings, "risk_score": score, "overall_risk": overall, "summary": summary, "recommendations": rec}

def cbc_rule_engine(r):
    score = 0
    findings = []
    def add(p, s, m, pts):
        nonlocal score
        findings.append({"parameter": p, "severity": s, "message": m})
        score += pts
    hb = r.get("Hemoglobin")
    if hb is not None and hb < 12: add("Hemoglobin", "HIGH", "Low hemoglobin may suggest anemia.", 15)
    wbc = r.get("WBC")
    if wbc is not None and (wbc > 11000 or wbc < 4000): add("WBC", "HIGH", "Abnormal WBC detected.", 15)
    plt = r.get("Platelets")
    if plt is not None and plt < 150: add("Platelets", "HIGH", "Low platelet count detected.", 15)
    overall = "CRITICAL" if score >= 50 else "HIGH" if score >= 30 else "MODERATE" if score >= 15 else "LOW"
    rec = []
    if hb is not None and hb < 12: rec.append("Check iron, vitamin B12, and anemia-related parameters.")
    if wbc is not None and (wbc > 11000 or wbc < 4000): rec.append("Discuss infection or immune concerns with a doctor.")
    if plt is not None and plt < 150: rec.append("Monitor bleeding/bruising symptoms and follow up.")
    if not rec: rec.append("CBC values look mostly stable. Continue routine monitoring.")
    summary = {
        "CRITICAL": "Critical blood/CBC risk pattern detected.",
        "HIGH": "High blood/CBC risk pattern detected.",
        "MODERATE": "Some blood/CBC concerns are present.",
        "LOW": "No major blood/CBC risk pattern detected.",
    }[overall]
    return {"findings": findings, "risk_score": score, "overall_risk": overall, "summary": summary, "recommendations": rec}

def route_and_predict(report_type, fields):
    if report_type == "diabetes":
        return diabetes_rule_engine(fields), predict_pipeline_model(MODELS["diabetes_model"], fields)
    if report_type == "liver":
        return liver_rule_engine(fields), predict_pipeline_model(MODELS["liver_model"], fields)
    if report_type == "heart":
        return heart_rule_engine(fields), predict_pipeline_model(MODELS["heart_model"], fields)
    if report_type == "cbc":
        return cbc_rule_engine(fields), predict_cbc_model(MODELS, fields)
    return None, None

## 7) Final summary builder

In [17]:
def build_summary(report_type, fields, rule_result, model_result):
    lines = []
    lines.append("=========== AI REPORT SUMMARY ===========")
    lines.append("")
    lines.append(f"Report Type: {report_type.upper()}")
    lines.append(report_type_reason(report_type))
    lines.append("")

    if model_result is not None:
        pred = model_result.get("prediction")
        prob = model_result.get("probability")
        lines.append("AI Model Prediction:")
        lines.append("- Elevated risk detected." if pred == 1 else "- Lower risk detected.")
        if prob is not None:
            lines.append(f"- Risk Probability: {prob}%")
        lines.append("")

    lines.append("Extracted Values:")
    if fields:
        for k, v in fields.items():
            lines.append(f"- {k}: {v}")
    else:
        lines.append("- No structured values extracted.")
    lines.append("")

    lines.append("Abnormal Findings:")
    if rule_result and rule_result["findings"]:
        for item in rule_result["findings"]:
            lines.append(f"- {item['parameter']} [{item['severity']}] -> {item['message']}")
    else:
        lines.append("- No major abnormal findings detected.")
    lines.append("")

    lines.append("Future Health Forecast:")
    if rule_result and rule_result["overall_risk"] in ["HIGH", "CRITICAL"]:
        lines.append("- Monitoring and follow-up are important.")
    else:
        lines.append("- No major future complication pattern detected.")
    lines.append("")

    lines.append("Recommendations:")
    if rule_result:
        for rec in rule_result["recommendations"]:
            lines.append(f"- {rec}")
    else:
        lines.append("- No recommendations available.")
    lines.append("")

    if rule_result and rule_result["overall_risk"] == "CRITICAL":
        lines.append("⚠ IMPORTANT WARNING: One or more values are critically abnormal.")
        lines.append("Please seek medical review as soon as possible.")
        lines.append("")

    lines.append("Simple Takeaway:")
    if rule_result:
        lines.append(rule_result["summary"])
    else:
        lines.append("The system could not confidently analyze this report.")
    lines.append("")
    lines.append("This AI-generated summary is for educational and supportive analysis only and does not replace medical advice.")
    return " ".join(lines)

def save_summary_text(summary_text, filename="report_summary.txt"):
    out = OUTPUTS_DIR / filename
    out.write_text(summary_text, encoding="utf-8")
    return str(out)

## 8) Main analysis pipeline

In [18]:
def analyze_report(file_path):
    file_path = str(file_path)
    suffix = Path(file_path).suffix.lower()

    if suffix == ".pdf":
        raw_text = extract_text_from_pdf(file_path)
    elif suffix in [".png", ".jpg", ".jpeg", ".tiff", ".bmp", ".webp"]:
        raw_text = extract_text_from_image(file_path)
    elif suffix == ".txt":
        raw_text = clean_text(Path(file_path).read_text(encoding="utf-8", errors="ignore"))
    else:
        raise ValueError("Unsupported file type. Use PDF, image, or TXT.")

    if not raw_text:
        return {
            "report_type": "unknown",
            "raw_text": "",
            "fields": {},
            "rule_result": None,
            "model_result": None,
            "summary": "No text could be extracted from the provided file."
        }

    report_type = detect_report_type(raw_text)
    fields = extract_report_fields(raw_text, report_type)
    rule_result, model_result = route_and_predict(report_type, fields)
    summary = build_summary(report_type, fields, rule_result, model_result)

    return {
        "report_type": report_type,
        "raw_text": raw_text,
        "fields": fields,
        "rule_result": rule_result,
        "model_result": model_result,
        "summary": summary,
    }

print("Main pipeline ready.")

Main pipeline ready.


## 9) Test with the sample PDF

In [19]:
sample_pdf = PROJECT_DIR / "sample_report.pdf"

if sample_pdf.exists():
    result = analyze_report(sample_pdf)
    print(result["summary"])
    saved = save_summary_text(result["summary"], "sample_report_summary.txt")
    print("\nSaved summary to:", saved)
else:
    print("sample_report.pdf not found in the current folder.")

Prediction error: feature_names mismatch: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'] ['Glucose', 'BMI', 'BloodPressure', 'Insulin', 'Age']
expected Pregnancies, SkinThickness, DiabetesPedigreeFunction in input data
=========== AI REPORT SUMMARY ===========  Report Type: DIABETES Glucose / insulin terms detected.  Extracted Values: - Glucose: 220.0 - BMI: 36.0 - BloodPressure: 150.0 - Insulin: 180.0 - Age: 1  Abnormal Findings: - Glucose [CRITICAL] -> Very high glucose detected. - BMI [MODERATE] -> High BMI detected. - Blood Pressure [HIGH] -> High blood pressure detected. - Insulin [MODERATE] -> Elevated insulin detected.  Future Health Forecast: - Monitoring and follow-up are important.  Recommendations: - Monitor blood sugar and reduce sugar intake. - Exercise and maintain a healthy diet. - Monitor blood pressure and reduce salt intake.  ⚠ IMPORTANT WARNING: One or more values are critically abnormal. Please seek 

## 10) What comes next
After this notebook is working, move to the frontend pages.
The frontend should upload a report, call this analyzer, and display the summary in a clean dashboard.


# ⭐ Advanced Production-Level Improvements

This section adds:

1. Multi-report detection  
2. Missing value handling  
3. OCR quality scoring  
4. Structured JSON output  
5. Emergency alert engine  

These improvements make the system frontend-ready and much more reliable for real-world healthcare report analysis.


In [20]:

# =========================================================
# ADVANCED REPORT ANALYZER IMPROVEMENTS
# =========================================================

NORMAL_RANGES = {
    "Glucose": "70-99 mg/dL",
    "BMI": "18.5-24.9",
    "BloodPressure": "90-120",
    "Hemoglobin": "12-16 g/dL",
    "WBC": "4000-11000",
    "Platelets": "150-450",
    "Total_Bilirubin": "0.1-1.2",
    "Albumin": "3.5-5.0"
}

REQUIRED_FIELDS = {
    "diabetes": ["Glucose", "BMI", "BloodPressure"],
    "heart": ["chol", "trestbps"],
    "liver": ["Total_Bilirubin", "Albumin"],
    "cbc": ["Hemoglobin", "WBC", "Platelets"]
}

# =========================================================
# MULTI REPORT DETECTION
# =========================================================

def detect_multiple_report_types(text):

    text = text.lower()

    detected = []

    diabetes_keywords = [
        "glucose",
        "blood sugar",
        "insulin",
        "hba1c"
    ]

    heart_keywords = [
        "cholesterol",
        "ecg",
        "angina",
        "heart rate",
        "troponin"
    ]

    liver_keywords = [
        "bilirubin",
        "alt",
        "ast",
        "albumin",
        "sgpt",
        "sgot"
    ]

    cbc_keywords = [
        "hemoglobin",
        "wbc",
        "rbc",
        "platelet",
        "mcv"
    ]

    if any(k in text for k in diabetes_keywords):
        detected.append("diabetes")

    if any(k in text for k in heart_keywords):
        detected.append("heart")

    if any(k in text for k in liver_keywords):
        detected.append("liver")

    if any(k in text for k in cbc_keywords):
        detected.append("cbc")

    if not detected:
        detected.append("unknown")

    return detected

# =========================================================
# OCR QUALITY SCORE
# =========================================================

def calculate_ocr_quality(text):

    words = len(text.split())

    if words >= 400:
        return 95

    elif words >= 250:
        return 85

    elif words >= 120:
        return 75

    elif words >= 60:
        return 60

    return 40

# =========================================================
# MISSING VALUE DETECTION
# =========================================================

def detect_missing_values(report_type, fields):

    missing = []

    required = REQUIRED_FIELDS.get(report_type, [])

    for item in required:

        if item not in fields:
            missing.append(item)

    return missing

# =========================================================
# EMERGENCY ALERT ENGINE
# =========================================================

def emergency_alert_engine(fields):

    alerts = []

    glucose = fields.get("Glucose")

    if glucose and glucose >= 300:

        alerts.append(
            "Extremely high glucose detected. Emergency diabetic risk possible."
        )

    bilirubin = fields.get("Total_Bilirubin")

    if bilirubin and bilirubin >= 5:

        alerts.append(
            "Very high bilirubin detected. Urgent liver evaluation recommended."
        )

    bp = fields.get("BloodPressure")

    if bp and bp >= 180:

        alerts.append(
            "Critically high blood pressure detected."
        )

    return alerts

# =========================================================
# CONFIDENCE ENGINE
# =========================================================

def confidence_level(probability):

    if probability is None:
        return "UNKNOWN"

    if probability >= 90:
        return "VERY HIGH"

    elif probability >= 75:
        return "HIGH"

    elif probability >= 55:
        return "MODERATE"

    return "LOW"

# =========================================================
# STRUCTURED JSON OUTPUT
# =========================================================

def create_structured_output(
    report_types,
    fields,
    rule_result,
    model_result,
    ocr_quality,
    missing_values,
    emergency_alerts
):

    probability = None

    if model_result:
        probability = model_result.get("probability")

    output = {

        "report_types": report_types,

        "ocr_quality_score": ocr_quality,

        "confidence_level": confidence_level(probability),

        "overall_risk": (
            rule_result["overall_risk"]
            if rule_result else "UNKNOWN"
        ),

        "risk_probability": probability,

        "findings": (
            rule_result["findings"]
            if rule_result else []
        ),

        "recommendations": (
            rule_result["recommendations"]
            if rule_result else []
        ),

        "missing_values": missing_values,

        "emergency_alerts": emergency_alerts,

        "extracted_values": fields
    }

    return output

# =========================================================
# IMPROVED FINAL SUMMARY
# =========================================================

def build_advanced_summary(
    report_types,
    fields,
    rule_result,
    model_result,
    ocr_quality,
    missing_values,
    emergency_alerts
):

    lines = []

    lines.append(
        "=========== ADVANCED AI REPORT SUMMARY ==========="
    )

    lines.append("")

    lines.append(
        f"Detected Report Types: {', '.join(report_types)}"
    )

    lines.append(
        f"OCR Quality Score: {ocr_quality}%"
    )

    if ocr_quality < 60:

        lines.append(
            "⚠ OCR quality is low. Some values may be missing or incorrect."
        )

    lines.append("")

    # -----------------------------------------------------
    # MODEL RESULT
    # -----------------------------------------------------

    if model_result:

        probability = model_result.get("probability")

        lines.append(
            f"Risk Probability: {probability}%"
        )

        lines.append(
            f"Confidence Level: {confidence_level(probability)}"
        )

    lines.append("")

    # -----------------------------------------------------
    # EXTRACTED VALUES
    # -----------------------------------------------------

    lines.append("Extracted Medical Values:")

    for key, value in fields.items():

        normal = NORMAL_RANGES.get(key, "Not Available")

        lines.append(
            f"- {key}: {value} | Normal Range: {normal}"
        )

    lines.append("")

    # -----------------------------------------------------
    # FINDINGS
    # -----------------------------------------------------

    lines.append("Medical Findings:")

    if rule_result and rule_result["findings"]:

        for item in rule_result["findings"]:

            lines.append(
                f"- {item['parameter']} [{item['severity']}] -> {item['message']}"
            )

    else:

        lines.append(
            "- No major abnormalities detected."
        )

    lines.append("")

    # -----------------------------------------------------
    # MISSING VALUES
    # -----------------------------------------------------

    if missing_values:

        lines.append(
            "Missing Important Values:"
        )

        for item in missing_values:

            lines.append(f"- {item}")

        lines.append("")

    # -----------------------------------------------------
    # EMERGENCY ALERTS
    # -----------------------------------------------------

    if emergency_alerts:

        lines.append(
            "🚨 EMERGENCY ALERTS:"
        )

        for item in emergency_alerts:

            lines.append(f"- {item}")

        lines.append("")

    # -----------------------------------------------------
    # RECOMMENDATIONS
    # -----------------------------------------------------

    lines.append("Recommendations:")

    if rule_result:

        for rec in rule_result["recommendations"]:

            lines.append(f"- {rec}")

    lines.append("")

    lines.append(
        "This AI-generated summary is for supportive analysis only."
    )

    return "\n".join(lines)

print("Advanced Report Analyzer Improvements Added Successfully.")


Advanced Report Analyzer Improvements Added Successfully.


In [21]:

# =========================================================
# TEST ADVANCED SYSTEM
# =========================================================

sample_fields = {

    "Glucose": 320,
    "BMI": 35,
    "BloodPressure": 185,
    "Total_Bilirubin": 5.6,
    "Albumin": 2.8

}

sample_text = '''
Glucose 320
BMI 35
Blood Pressure 185
Total Bilirubin 5.6
Albumin 2.8
'''

report_types = detect_multiple_report_types(sample_text)

ocr_quality = calculate_ocr_quality(sample_text)

missing_values = detect_missing_values(
    "diabetes",
    sample_fields
)

emergency_alerts = emergency_alert_engine(
    sample_fields
)

rule_result = {

    "overall_risk": "CRITICAL",

    "findings": [

        {
            "parameter": "Glucose",
            "severity": "CRITICAL",
            "message": "Very high glucose detected."
        }

    ],

    "recommendations": [

        "Immediate medical consultation recommended."
    ]
}

model_result = {

    "prediction": 1,
    "probability": 96.5
}

structured_output = create_structured_output(
    report_types,
    sample_fields,
    rule_result,
    model_result,
    ocr_quality,
    missing_values,
    emergency_alerts
)

advanced_summary = build_advanced_summary(
    report_types,
    sample_fields,
    rule_result,
    model_result,
    ocr_quality,
    missing_values,
    emergency_alerts
)

print(advanced_summary)

print("\n================ JSON OUTPUT ================\n")

print(structured_output)


=========== ADVANCED AI REPORT SUMMARY ===========

Detected Report Types: diabetes, liver
OCR Quality Score: 40%
⚠ OCR quality is low. Some values may be missing or incorrect.

Risk Probability: 96.5%
Confidence Level: VERY HIGH

Extracted Medical Values:
- Glucose: 320 | Normal Range: 70-99 mg/dL
- BMI: 35 | Normal Range: 18.5-24.9
- BloodPressure: 185 | Normal Range: 90-120
- Total_Bilirubin: 5.6 | Normal Range: 0.1-1.2
- Albumin: 2.8 | Normal Range: 3.5-5.0

Medical Findings:
- Glucose [CRITICAL] -> Very high glucose detected.

🚨 EMERGENCY ALERTS:
- Extremely high glucose detected. Emergency diabetic risk possible.
- Very high bilirubin detected. Urgent liver evaluation recommended.
- Critically high blood pressure detected.

Recommendations:
- Immediate medical consultation recommended.

This AI-generated summary is for supportive analysis only.

================ JSON OUTPUT ================

{'report_types': ['diabetes', 'liver'], 'ocr_quality_score': 40, 'confidence_level': 'VER